<img src="https://sfile.chatglm.cn/images-ppt/text_embeddings_cover.png" width="500" align="center">

# Обработка текста: эмбеддинги и токенизация - как машины "читают"

**Роль:** Преподаватель по ML
**Уровень:** средний (основы Python + понимание нейросетей)
**Время прохождения:** ~90-120 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **почему** компьютер не понимает текст "как есть";
- узнаете, что такое **токенизация** и какие бывают способы разрезания текста;
- **своими руками** реализуете TF-IDF - классический метод векторизации текста;
- разберёте Word2Vec и **визуализируете** эмбеддинги слов;
- поймёте BPE - алгоритм токенизации, который используют GPT и BERT;
- увидите, **как текст превращается в числа**, шаг за шагом;
- построите интерактивные демонстраторы с виджетами.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Почему текст - это проблема? | Компьютер видит только числа |
| 3 | Токенизация: режем текст на куски | Word-level, char-level, subword |
| 4 | Словарь и one-hot кодирование | Превращаем токены в векторы |
| 5 | TF-IDF своими руками | Классическая векторизация документов |
| 6 | Интерактивный TF-IDF | Виджеты: меняем текст, смотрим веса |
| 7 | От one-hot к эмбеддингам | Почему one-hot не хватает |
| 8 | Word2Vec: идея | Слова с похожим контекстом - похожие векторы |
| 9 | Word2Vec: визуализация | t-SNE и интерактивный 3D-график |
| 10 | BPE: Byte-Pair Encoding | Как токенизируют GPT и BERT |
| 11 | BPE: интерактивный демонстратор | Меняем параметры, смотрим результат |
| 12 | Сравнение подходов | One-hot vs TF-IDF vs Word2Vec vs BPE |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **numpy** | Работа с массивами и матрицами |
| **matplotlib** | Визуализация графиков и диаграмм |
| **sklearn** | TF-IDF и t-SNE для сравнения |
| **ipywidgets** | Интерактивные виджеты |
| **collections** | Счётчики и частотные словари |

> В этом блокноте мы строим всё **с нуля** - где можно, без готовых библиотек!

In [ ]:
# ============================================================
# ШАГ 1: Импортируем все нужные библиотеки
# ============================================================

import numpy as np                          # основная библиотека для массивов и математики
import matplotlib.pyplot as plt             # для построения графиков и визуализаций
from matplotlib import cm                   # цветовые карты для красивых графиков
from collections import Counter             # счётчик частот - считаем, сколько раз слово встречается
import re                                   # регулярные выражения - для очистки текста
import math                                 # математические функции (log, sqrt)
from itertools import combinations          # генерация комбинаций (для пар слов)
from IPython.display import display, HTML   # красивый вывод в ноутбуке
import ipywidgets as widgets                # интерактивные виджеты (слайдеры, кнопки)
from ipywidgets import interact, interactive, fixed  # декораторы для интерактива

# Настройка matplotlib для красивых графиков
plt.rcParams['figure.figsize'] = (10, 6)    # размер графиков по умолчанию
plt.rcParams['font.size'] = 12              # размер шрифта
plt.rcParams['axes.grid'] = True            # показываем сетку на графиках

print("Все библиотеки импортированы!")

---
## Шаг 2. Почему текст - это проблема?

Компьютер работает с **числами**. Текст для него - просто последовательность байтов.
Нейросеть тоже принимает на вход **числа**. Значит, нам нужно превратить текст в числа.

Но как? Есть несколько проблем:

1. **Слова не числа** - "кот" и "собака" нельзя просто сложить
2. **Синонимы** - "большой" и "огромный" означают похожее, но выглядят по-разному
3. **Контекст** - "банк" может означать финансовое учреждение или берег реки
4. **Длина** - предложения разной длины, а нейросеть хочет фиксированный вход

Давайте посмотрим, как компьютер "видит" текст без обработки:

In [ ]:
# ============================================================
# ШАГ 2: Как компьютер "видит" текст без обработки
# ============================================================

# Возьмём три предложения для примера
sentence_1 = "Кот сидит на коврике"                   # первое предложение
sentence_2 = "Собака лежит на коврике"                # второе предложение
sentence_3 = "Кот сидит на диване"                    # третье предложение

# Попробуем "наивный" подход: просто переведём символы в числа (Unicode)
chars_1 = [ord(c) for c in sentence_1]               # ord() даёт Unicode-код символа
chars_2 = [ord(c) for c in sentence_2]               # то же для второго предложения
chars_3 = [ord(c) for c in sentence_3]               # то же для третьего предложения

print("Предложение 1:", sentence_1)
print("Unicode-коды :", chars_1)
print()
print("Предложение 2:", sentence_2)
print("Unicode-коды :", chars_2)
print()
print("Предложение 3:", sentence_3)
print("Unicode-коды :", chars_3)
print()
print("ПРОБЛЕМА: эти числа ничего не говорят о СМЫСЛЕ текста!")
print("   'Кот' и 'кот' - разные наборы чисел, хотя это одно слово")
print("   'Коврик' в предложениях 1 и 2 - одно и то же, но позиции разные")

### Визуализируем проблему

Покажем, что Unicode-коды не передают сходство слов:

In [ ]:
# ============================================================
# Визуализация: Unicode-коды не передают смысл
# ============================================================

# Слова для сравнения
words = ["кот", "кошка", "собака", "коврик", "диван"]  # 5 слов для примера

# Получаем средний Unicode-код каждого слова
avg_codes = [np.mean([ord(c) for c in word]) for word in words]  # средний код символов

# Рисуем столбчатую диаграмму
fig, ax = plt.subplots(figsize=(8, 5))                # создаём фигуру
colors = ['#FF6B6B', '#FF6B6B', '#4ECDC4', '#45B7D1', '#45B7D1']  # цвета: похожие слова - один цвет
bars = ax.bar(words, avg_codes, color=colors, edgecolor='black', linewidth=1.2)  # рисуем столбцы

# Добавляем значения над столбцами
for bar, val in zip(bars, avg_codes):                  # для каждого столбца
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,  # позиция текста
            f'{val:.0f}', ha='center', va='bottom', fontsize=11)  # само значение

ax.set_ylabel("Средний Unicode-код", fontsize=13)     # подпись оси Y
ax.set_title("Unicode-коды НЕ отражают сходство слов", fontsize=14, fontweight='bold')  # заголовок
ax.set_ylim(0, max(avg_codes) + 100)                  # пределы оси Y

# Добавляем легенду-пояснение
ax.annotate("Похожие слова\n(кот = кошка)", xy=(0.5, avg_codes[0]),
            xytext=(2.5, avg_codes[0]+80), fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))  # стрелка

plt.tight_layout()                                    # чтобы не обрезалось
plt.show()                                            # показываем график

print("ВНИМАНИЕ: 'кот' и 'кошка' - семантически похожи, но их Unicode-коды далеки!")
print("ВНИМАНИЕ: 'кот' и 'собака' - семантически дальше, но Unicode-коды ближе!")
print("   -> Нам нужен ДРУГОЙ способ представления текста")

---
## Шаг 3. Токенизация: режем текст на куски

**Токенизация** - это процесс разрезания текста на минимальные значимые единицы - **токены**.

Существует три основных подхода:

| Подход | Пример "Кот сидит" | Плюсы | Минусы |
|--------|---------------------|-------|--------|
| **Word-level** | ["Кот", "сидит"] | Понятный, простой | Огромный словарь, неизвестные слова |
| **Char-level** | ["К","о","т"," ","с","и","д","и","т"] | Маленький словарь | Длинные последовательности, нет смысла |
| **Subword (BPE)** | ["Кот", "сид", "ит"] | Баланс: меньше словарь + есть смысл | Сложнее алгоритм |

Давайте реализуем каждый подход:

In [ ]:
# ============================================================
# ШАГ 3a: Word-level токенизация - по словам
# ============================================================

def tokenize_word(text):
    # Word-level токенизация: разбиваем текст на слова.
    #
    # Алгоритм:
    # 1. Приводим к нижнему регистру (чтобы 'Кот' и 'кот' = одно слово)
    # 2. Убираем пунктуацию
    # 3. Разбиваем по пробелам
    #
    # Args:
    #   text (str): входной текст
    # Returns:
    #   list: список токенов-слов

    # Шаг 1: нижний регистр - чтобы слова с большой буквы совпадали
    text_lower = text.lower()                          # "Кот сидит" -> "кот сидит"

    # Шаг 2: убираем пунктуацию с помощью регулярного выражения
    # [^\w\s] означает "всё, что НЕ буква/цифра и НЕ пробел"
    text_clean = re.sub(r'[^\w\s]', '', text_lower)  # "кот, сидит!" -> "кот сидит"

    # Шаг 3: разбиваем по пробелам
    tokens = text_clean.split()                        # "кот сидит" -> ["кот", "сидит"]

    return tokens                                      # возвращаем список токенов


# Тестируем на примерах
test_text = "Кот сидит на коврике. Собака лежит на диване!"  # тестовый текст
word_tokens = tokenize_word(test_text)                # токенизируем
print("Исходный текст:", test_text)                   # покажем оригинал
print("Word токены  :", word_tokens)                  # покажем результат
print("Количество токенов:", len(word_tokens))        # сколько токенов

In [ ]:
# ============================================================
# ШАГ 3b: Char-level токенизация - по символам
# ============================================================

def tokenize_char(text):
    # Char-level токенизация: разбиваем текст на отдельные символы.
    # Простейший подход - каждый символ = один токен.
    # Пробелы тоже являются токенами!
    #
    # Args:
    #   text (str): входной текст
    # Returns:
    #   list: список токенов-символов

    # Просто превращаем строку в список символов
    tokens = list(text)                                # "Кот" -> ["К", "о", "т"]

    return tokens                                      # возвращаем список символов


# Тестируем на том же тексте
char_tokens = tokenize_char(test_text)                 # токенизируем посимвольно
print("Исходный текст:", test_text)                    # оригинал
print("Char токены  :", char_tokens)                   # результат
print("Количество токенов:", len(char_tokens))         # сколько токенов
print()
print("Char-level даёт много токенов, но каждый токен - это всего один символ")
print("Нейросети сложнее уловить смысл из отдельных букв")

In [ ]:
# ============================================================
# ШАГ 3c: Визуальное сравнение подходов токенизации
# ============================================================

# Возьмём предложение для сравнения
sample = "Машинное обучение это круто"               # тестовое предложение

# Получаем токены обоими способами
word_toks = tokenize_word(sample)                      # по словам
char_toks = tokenize_char(sample)                      # по символам

# Для наглядности ограничим char-токены (их слишком много)
char_toks_display = char_toks[:20]                     # показываем первые 20 символов

# Создаём визуализацию - цветные блоки для токенов
fig, axes = plt.subplots(2, 1, figsize=(14, 5))       # 2 строки, 1 столбец

# --- Word-level ---
ax = axes[0]                                           # первая ось
colors_word = plt.cm.Set3(np.linspace(0, 1, len(word_toks)))  # цвета для слов
for i, (token, color) in enumerate(zip(word_toks, colors_word)):  # для каждого токена
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, facecolor=color,  # рисуем прямоугольник
                               edgecolor='black', linewidth=1.5))
    ax.text(i + 0.45, 0.5, token, ha='center', va='center',      # текст токена
            fontsize=11, fontweight='bold')
ax.set_xlim(-0.1, len(word_toks))                     # пределы оси X
ax.set_ylim(-0.2, 1.3)                                # пределы оси Y
ax.set_title(f"Word-level: {len(word_toks)} токенов", fontsize=13, fontweight='bold')  # заголовок
ax.axis('off')                                         # убираем оси

# --- Char-level ---
ax = axes[1]                                           # вторая ось
colors_char = plt.cm.Pastel1(np.linspace(0, 1, len(char_toks_display)))  # цвета
for i, (token, color) in enumerate(zip(char_toks_display, colors_char)):  # для каждого символа
    ax.add_patch(plt.Rectangle((i * 0.5, 0), 0.45, 1, facecolor=color,  # прямоугольник
                               edgecolor='black', linewidth=1))
    ax.text(i * 0.5 + 0.225, 0.5, token, ha='center', va='center',  # текст
            fontsize=9)
ax.set_xlim(-0.1, len(char_toks_display) * 0.5)       # пределы
ax.set_ylim(-0.2, 1.3)                                # пределы
ax.set_title(f"Char-level: {len(char_toks)} токенов (показано {len(char_toks_display)})",  # заголовок
             fontsize=13, fontweight='bold')
ax.axis('off')                                         # убираем оси

plt.suptitle("Сравнение способов токенизации", fontsize=15, fontweight='bold', y=1.02)  # общий заголовок
plt.tight_layout()                                     # чтобы не обрезалось
plt.show()                                             # показываем

print(f"Word-level: {len(word_toks)} токенов")
print(f"Char-level: {len(char_toks)} токенов  <- в {len(char_toks)/len(word_toks):.1f} раз больше!")
print()
print("Word-level компактнее, но словарь может быть огромным (десятки тысяч слов)")
print("Char-level маленький словарь, но очень длинные последовательности")

---
## Шаг 4. Словарь и one-hot кодирование

После токенизации нужен **словарь** - отображение каждого уникального токена в число (индекс).

**One-hot кодирование** превращает каждый индекс в вектор из нулей и одной единицы:

```
Словарь: {"кот": 0, "сидит": 1, "на": 2, "коврике": 3}

"кот"     -> [1, 0, 0, 0]
"сидит"   -> [0, 1, 0, 0]
"на"      -> [0, 0, 1, 0]
"коврике" -> [0, 0, 0, 1]
```

Проблема: все векторы **ортогональны** - ни одно слово не "похоже" на другое!

In [ ]:
# ============================================================
# ШАГ 4: Строим словарь и one-hot векторы
# ============================================================

def build_vocabulary(token_lists):
    # Строим словарь из списков токенов.
    #
    # Алгоритм:
    # 1. Собираем все уникальные токены
    # 2. Сортируем по частоте (самые частые - первые)
    # 3. Назначаем каждому токену индекс
    #
    # Args:
    #   token_lists: список списков токенов (один список = один документ)
    # Returns:
    #   dict: токен -> индекс
    #   dict: индекс -> токен (обратный словарь)

    # Собираем все токены в один плоский список
    all_tokens = []                                    # пустой список
    for token_list in token_lists:                     # для каждого документа
        all_tokens.extend(token_list)                  # добавляем все токены

    # Считаем частоту каждого токена
    token_counts = Counter(all_tokens)                 # Counter: {"кот": 3, "сидит": 2, ...}

    # Сортируем по частоте (убывание) и назначаем индексы
    sorted_tokens = token_counts.most_common()         # список (токен, частота) по убыванию

    # Строим словари: токен -> индекс и индекс -> токен
    token2idx = {}                                     # прямой словарь: токен -> индекс
    idx2token = {}                                     # обратный словарь: индекс -> токен
    for idx, (token, count) in enumerate(sorted_tokens):  # для каждого токена с его частотой
        token2idx[token] = idx                         # прямой: "кот" -> 0
        idx2token[idx] = token                         # обратный: 0 -> "кот"

    return token2idx, idx2token                        # возвращаем оба словаря


def one_hot_encode(token, token2idx):
    # Кодируем один токен в one-hot вектор.
    # One-hot - это вектор длины |V| (размер словаря),
    # где на позиции индекса токена стоит 1, а везде 0.
    #
    # Args:
    #   token (str): токен для кодирования
    #   token2idx (dict): словарь токен -> индекс
    # Returns:
    #   np.array: one-hot вектор

    vocab_size = len(token2idx)                        # размер словаря = длина вектора
    vector = np.zeros(vocab_size)                      # вектор из нулей
    if token in token2idx:                             # если токен есть в словаре
        idx = token2idx[token]                         # получаем его индекс
        vector[idx] = 1                                # ставим 1 на позиции индекса
    return vector                                      # возвращаем вектор


# === Демонстрация ===

# Корпус из нескольких предложений
corpus = [
    "кот сидит на коврике",
    "собака лежит на коврике",
    "кот лежит на диване",
    "собака сидит на стуле",
    "кот и собака играют на коврике"
]

# Токенизируем все предложения
tokenized_corpus = [tokenize_word(doc) for doc in corpus]  # список списков токенов

# Строим словарь
token2idx, idx2token = build_vocabulary(tokenized_corpus)  # прямые и обратные словари

print("Словарь (токен -> индекс):")
for token, idx in sorted(token2idx.items(), key=lambda x: x[1]):  # по порядку индексов
    print(f"  {token:12s} -> {idx}")                   # красивый вывод

print(f"\nРазмер словаря: {len(token2idx)} токенов")

# Кодируем несколько слов в one-hot
print("\nOne-hot кодирование:")
for word in ["кот", "собака", "коврик"]:              # три примера
    if word in token2idx:                              # если слово в словаре
        vec = one_hot_encode(word, token2idx)          # получаем one-hot вектор
        non_zero = np.where(vec == 1)[0][0]            # позиция единицы
        print(f"  '{word}' -> единица на позиции {non_zero}, длина вектора = {len(vec)}")

In [ ]:
# ============================================================
# Визуализация: one-hot векторы - все ортогональны
# ============================================================

# Выберем несколько слов для визуализации
words_to_show = ["кот", "собака", "сидит", "лежит", "на", "коврике"]  # 6 слов

# Собираем их one-hot векторы в матрицу
one_hot_matrix = np.array([one_hot_encode(w, token2idx) for w in words_to_show])  # матрица 6x|V|

# Рисуем heatmap
fig, ax = plt.subplots(figsize=(14, 5))                # создаём фигуру

# Показываем только столбцы, где есть хотя бы одна единица
active_cols = np.where(one_hot_matrix.sum(axis=0) > 0)[0]  # столбцы с единицами
matrix_display = one_hot_matrix[:, active_cols]        # вырезаем нужные столбцы

# Подписи столбцов - имена токенов
col_labels = [idx2token.get(active_cols[i], f"col_{i}") for i in range(len(active_cols))]  # имена

im = ax.imshow(matrix_display, cmap='Blues', aspect='auto', interpolation='nearest')  # heatmap
ax.set_xticks(range(len(col_labels)))                  # позиции столбцов
ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=11)  # подписи столбцов
ax.set_yticks(range(len(words_to_show)))               # позиции строк
ax.set_yticklabels(words_to_show, fontsize=12)         # подписи строк

# Подписываем единицы на матрице
for i in range(len(words_to_show)):                    # по строкам
    for j in range(len(active_cols)):                  # по столбцам
        if matrix_display[i, j] == 1:                  # если это единица
            ax.text(j, i, '1', ha='center', va='center',  # подписываем
                    fontsize=14, fontweight='bold', color='red')

ax.set_title("One-hot кодирование: каждое слово - одинокая единица в море нулей",  # заголовок
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("ПРОБЛЕМА one-hot:")
print("   1. Векторы огромные (длина = размер словаря)")
print("   2. Все слова одинаково 'далеки' друг от друга (косинусная близость = 0)")
print("   3. Не передают семантическое сходство ('кот' ~ 'собака' - но векторы ортогональны)")
print("   4. Разреженные - почти все элементы = 0")

---
## Шаг 5. TF-IDF своими руками

**TF-IDF** (Term Frequency - Inverse Document Frequency) - классический метод взвешивания слов в документе.

Идея:
- **TF** (Term Frequency): насколько часто слово встречается в **данном документе**
- **IDF** (Inverse Document Frequency): насколько **редкое** слово во всём корпусе
- Редкие, но важные слова получают высокий вес; частые и бесполезные (например, "на", "и") - низкий

Формулы:
- **TF(t, d)** = (количество слова t в документе d) / (всего слов в d)
- **IDF(t)** = log( N / (количество документов, содержащих t) )
- **TF-IDF(t, d)** = TF(t, d) x IDF(t)

Строим **с нуля**, без sklearn!

In [ ]:
# ============================================================
# ШАГ 5a: TF - Term Frequency: частота слова в документе
# ============================================================

def compute_tf(tokens):
    # Вычисляем TF (Term Frequency) для списка токенов.
    #
    # TF(t, d) = count(t in d) / len(d)
    #
    # Где:
    #   t - токен (слово)
    #   d - документ (список токенов)
    #   count(t in d) - сколько раз токен встречается в документе
    #   len(d) - общее количество токенов в документе
    #
    # Args:
    #   tokens (list): список токенов документа
    # Returns:
    #   dict: токен -> TF-значение (от 0 до 1)

    token_counts = Counter(tokens)                      # считаем частоту каждого токена
    total_tokens = len(tokens)                          # всего токенов в документе

    tf = {}                                            # пустой словарь для TF
    for token, count in token_counts.items():           # для каждого токена
        tf[token] = count / total_tokens                # TF = частота / общее количество

    return tf                                           # возвращаем словарь TF


# Вычисляем TF для первого документа нашего корпуса
doc_tokens = tokenized_corpus[0]                       # токены первого документа
tf_doc0 = compute_tf(doc_tokens)                        # TF для первого документа

print(f"Документ: '{corpus[0]}'")
print(f"TF-значения:")
for token, tf_val in sorted(tf_doc0.items(), key=lambda x: -x[1]):  # по убыванию
    bar = "#" * int(tf_val * 40)                        # визуальная шкала
    print(f"  {token:12s} TF = {tf_val:.3f}  {bar}")   # красивый вывод

In [ ]:
# ============================================================
# ШАГ 5b: IDF - Inverse Document Frequency: редкость слова
# ============================================================

def compute_idf(tokenized_docs):
    # Вычисляем IDF (Inverse Document Frequency) для всех токенов корпуса.
    #
    # IDF(t) = log( N / df(t) )
    #
    # Где:
    #   N - общее количество документов
    #   df(t) - сколько документов содержат токен t
    #   log - натуральный логарифм
    #
    # IDF высокий -> слово редкое (информативное)
    # IDF низкий  -> слово частое (малоинформативное, например предлоги)
    #
    # Args:
    #   tokenized_docs (list): список списков токенов
    # Returns:
    #   dict: токен -> IDF-значение

    N = len(tokenized_docs)                            # количество документов

    # Считаем df(t): в скольких документах встречается каждый токен
    df = {}                                            # пустой словарь document frequency
    for doc_tokens in tokenized_docs:                  # для каждого документа
        unique_tokens = set(doc_tokens)                # уникальные токены документа
        for token in unique_tokens:                    # для каждого уникального токена
            df[token] = df.get(token, 0) + 1           # увеличиваем счётчик на 1

    # Вычисляем IDF для каждого токена
    idf = {}                                           # пустой словарь для IDF
    for token, doc_count in df.items():                # для каждого токена
        idf[token] = math.log(N / doc_count)           # IDF = log(N / df)

    return idf                                          # возвращаем словарь IDF


# Вычисляем IDF для нашего корпуса
idf_values = compute_idf(tokenized_corpus)              # IDF для всех токенов

print("IDF-значения (во всём корпусе):")
for token, idf_val in sorted(idf_values.items(), key=lambda x: -x[1]):  # по убыванию
    bar = "#" * int(idf_val * 10)                       # визуальная шкала
    print(f"  {token:12s} IDF = {idf_val:.3f}  {bar}")

print()
print("'на' встречается во ВСЕХ документах -> IDF = 0 (бесполезное слово)")
print("'играют' только в одном документе -> IDF высокий (информативное слово)")

In [ ]:
# ============================================================
# ШАГ 5c: TF-IDF - объединяем TF и IDF
# ============================================================

def compute_tfidf(tokenized_docs):
    # Вычисляем TF-IDF для всех документов.
    #
    # TF-IDF(t, d) = TF(t, d) * IDF(t)
    #
    # Результат: матрица размером (документы x токены),
    # где каждая ячейка - вес токена в документе.
    #
    # Args:
    #   tokenized_docs (list): список списков токенов
    # Returns:
    #   np.array: TF-IDF матрица
    #   dict: токен -> индекс столбца

    # Строим общий словарь по всему корпусу
    token2idx_tfidf, _ = build_vocabulary(tokenized_docs)  # словарь

    # Вычисляем IDF (один раз для всего корпуса)
    idf = compute_idf(tokenized_docs)                   # IDF-значения

    # Создаём матрицу TF-IDF
    num_docs = len(tokenized_docs)                      # количество документов
    vocab_size = len(token2idx_tfidf)                   # размер словаря
    tfidf_matrix = np.zeros((num_docs, vocab_size))     # матрица из нулей

    # Заполняем матрицу
    for doc_idx, doc_tokens in enumerate(tokenized_docs):  # для каждого документа
        tf = compute_tf(doc_tokens)                      # TF для этого документа
        for token, tf_val in tf.items():                 # для каждого токена
            col_idx = token2idx_tfidf[token]             # столбец токена
            tfidf_matrix[doc_idx, col_idx] = tf_val * idf.get(token, 0)  # TF * IDF

    return tfidf_matrix, token2idx_tfidf                 # возвращаем матрицу и словарь


# Вычисляем TF-IDF для нашего корпуса
tfidf_matrix, tfidf_vocab = compute_tfidf(tokenized_corpus)  # матрица TF-IDF

print(f"Форма TF-IDF матрицы: {tfidf_matrix.shape}")
print(f"   (строки = документы, столбцы = токены)")
print()
print("TF-IDF для первого документа:")
doc0_tokens = tokenized_corpus[0]                      # токены первого документа
tf0 = compute_tf(doc0_tokens)                           # TF первого документа
for token in doc0_tokens:                               # для каждого токена
    idx = tfidf_vocab[token]                            # индекс в матрице
    tfidf_val = tfidf_matrix[0, idx]                    # значение TF-IDF
    print(f"  {token:12s} TF={tf0[token]:.3f}  IDF={idf_values.get(token, 0):.3f}  TF-IDF={tfidf_val:.4f}")

In [ ]:
# ============================================================
# ШАГ 5d: Визуализация TF-IDF матрицы (heatmap)
# ============================================================

# Берём только токены с ненулевыми TF-IDF значениями (убираем шум)
# Для визуализации ограничим топ-15 самых важных токенов
token_importance = np.sum(tfidf_matrix, axis=0)        # сумма TF-IDF по всем документам
top_indices = np.argsort(token_importance)[-15:][::-1]  # топ-15 по важности

# Создаём укороченную матрицу для визуализации
tfidf_display = tfidf_matrix[:, top_indices]           # вырезаем столбцы
display_tokens = [list(tfidf_vocab.keys())[list(tfidf_vocab.values()).index(i)]  # имена токенов
                  for i in top_indices]

# Рисуем heatmap
fig, ax = plt.subplots(figsize=(16, 6))                # создаём фигуру
im = ax.imshow(tfidf_display, cmap='YlOrRd', aspect='auto')  # тепловая карта

# Подписи
ax.set_xticks(range(len(display_tokens)))              # позиции столбцов
ax.set_xticklabels(display_tokens, rotation=45, ha='right', fontsize=11)  # имена токенов
ax.set_yticks(range(len(corpus)))                      # позиции строк
doc_labels = [f"Док {i+1}" for i in range(len(corpus))]  # имена документов
ax.set_yticklabels(doc_labels, fontsize=11)            # подписи строк

# Числа в ячейках
for i in range(len(corpus)):                           # по документам
    for j in range(len(display_tokens)):               # по токенам
        val = tfidf_display[i, j]                      # значение ячейки
        if val > 0:                                    # если не ноль
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',  # подписываем
                    fontsize=8, color='white' if val > 0.1 else 'black',  # цвет текста
                    fontweight='bold')

plt.colorbar(im, ax=ax, label='TF-IDF вес')            # цветовая шкала
ax.set_title("TF-IDF матрица: какие слова важны в каких документах",  # заголовок
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Тёмные клетки = высокие TF-IDF веса -> слово важно для этого документа")
print("'на' - светлое везде (низкий IDF, т.к. есть во всех документах)")
print("'играют' - тёмное только в одном документе (редкое, информативное)")

---
## Шаг 6. Интерактивный TF-IDF

Попробуйте **ввести свой текст** и увидеть TF-IDF в реальном времени!

In [ ]:
# ============================================================
# ШАГ 6: Интерактивный TF-IDF калькулятор с виджетами
# ============================================================

# Готовим корпус из более осмысленных текстов
sample_corpus = [
    "машинное обучение это область искусственного интеллекта",
    "глубокое обучение использует нейронные сети с многими слоями",
    "обработка естественного языка работает с текстовыми данными",
    "компьютерное зрение распознаёт объекты на изображениях",
    "рекомендательные системы предлагают персональные рекомендации",
    "машинное обучение применяется в обработке языка и зрения"
]

# Токенизируем
tokenized_sample = [tokenize_word(doc) for doc in sample_corpus]  # токены каждого документа

# Предвычисляем IDF (он не меняется при вводе нового документа)
idf_sample = compute_idf(tokenized_sample)              # IDF по корпусу


def interactive_tfidf(new_text):
    # Вычисляем и визуализируем TF-IDF для пользовательского текста
    # относительно существующего корпуса.

    if not new_text.strip():                            # если пустой ввод
        print("Введите текст для анализа!")
        return

    # Токенизируем новый текст
    new_tokens = tokenize_word(new_text)                # токены нового текста

    if not new_tokens:                                  # если нет валидных токенов
        print("Нет валидных токенов! Попробуйте другой текст.")
        return

    # Вычисляем TF для нового текста
    tf_new = compute_tf(new_tokens)                     # TF нового документа

    # Вычисляем TF-IDF для нового текста (используя IDF корпуса)
    tfidf_new = {}                                     # словарь TF-IDF нового текста
    for token, tf_val in tf_new.items():                # для каждого токена
        idf_val = idf_sample.get(token, math.log(len(sample_corpus) + 1))  # IDF (или оценка)
        tfidf_new[token] = tf_val * idf_val             # TF-IDF = TF * IDF

    # Сортируем по TF-IDF
    sorted_tfidf = sorted(tfidf_new.items(), key=lambda x: -x[1])  # по убыванию

    # Рисуем столбчатую диаграмму
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))  # два графика рядом

    # --- TF-IDF веса ---
    tokens_sorted = [t for t, v in sorted_tfidf]        # токены
    values_sorted = [v for t, v in sorted_tfidf]        # значения
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(tokens_sorted)))  # цвета

    ax1.barh(tokens_sorted[::-1], values_sorted[::-1], color=colors[::-1],  # горизонтальные столбцы
             edgecolor='black', linewidth=0.5)
    ax1.set_xlabel("TF-IDF вес", fontsize=12)           # подпись оси
    ax1.set_title("TF-IDF веса вашего текста", fontsize=13, fontweight='bold')  # заголовок

    # --- Сравнение TF и IDF ---
    tf_vals = [tf_new.get(t, 0) for t in tokens_sorted]  # TF значения
    idf_vals = [idf_sample.get(t, 0) for t in tokens_sorted]  # IDF значения

    x = np.arange(len(tokens_sorted))                   # позиции
    width = 0.35                                        # ширина столбцов
    ax2.bar(x - width/2, tf_vals, width, label='TF', color='#FF6B6B', edgecolor='black')  # TF
    ax2.bar(x + width/2, idf_vals, width, label='IDF', color='#4ECDC4', edgecolor='black')  # IDF
    ax2.set_xticks(x)                                   # позиции
    ax2.set_xticklabels(tokens_sorted, rotation=45, ha='right')  # подписи
    ax2.legend(fontsize=12)                             # легенда
    ax2.set_title("TF vs IDF: что даёт вес слову?", fontsize=13, fontweight='bold')  # заголовок
    ax2.set_ylabel("Значение", fontsize=12)             # подпись оси

    plt.tight_layout()                                  # чтобы не обрезалось
    plt.show()                                          # показываем

    # Текстовый вывод
    print(f"\nВаш текст: '{new_text}'")
    print(f"Токены: {new_tokens}")
    print(f"Самое важное слово: '{sorted_tfidf[0][0]}' (TF-IDF = {sorted_tfidf[0][1]:.4f})")


# Создаём интерактивный виджет
text_input = widgets.Text(                              # текстовое поле
    value='машинное обучение и язык',                   # значение по умолчанию
    placeholder='Введите текст на русском...',           # подсказка
    description='Текст:',                               # подпись
    layout=widgets.Layout(width='500px')                # ширина
)

# Связываем виджет с функцией
interactive_plot = interactive(interactive_tfidf, new_text=text_input)  # интерактив
display(interactive_plot)                               # показываем виджет

print("Введите текст выше и нажмите Enter - увидите TF-IDF в реальном времени!")

---
## Шаг 7. От one-hot к эмбеддингам

Мы увидели, что one-hot векторы:
- **Огромные** (длина = размер словаря, могут быть десятки тысяч)
- **Разреженные** (почти все нули)
- **Не передают сходство** (все слова одинаково "далеки")

**Эмбеддинги** - это **плотные** векторы маленькой размерности (50-300), где:
- Похожие слова -> похожие векторы
- Можно складывать и вычитать: king - man + woman ~ queen
- Компактные: 300 чисел вместо 100 000

Как получить эмбеддинги? Один из первых и самых понятных методов - **Word2Vec**!

In [ ]:
# ============================================================
# ШАГ 7: Визуализация one-hot vs эмбеддинги
# ============================================================

# Создадим игрушечные эмбеддинги для демонстрации
# Представим, что у нас 3D-эмбеддинги (на практике 100-300 измерений)
toy_embeddings = {
    "кот":     np.array([0.9, 0.1, 0.2]),              # животное, маленькое, пушистое
    "кошка":   np.array([0.85, 0.05, 0.25]),            # очень похоже на кота
    "собака":  np.array([0.7, 0.8, 0.3]),               # животное, большое, пушистое
    "щенок":   np.array([0.6, 0.3, 0.35]),              # животное, маленькое
    "коврик":  np.array([0.1, 0.1, 0.9]),               # не животное, предмет
    "диван":   np.array([0.05, 0.15, 0.85]),            # не животное, предмет
    "сидит":   np.array([0.2, 0.2, 0.1]),               # действие
    "лежит":   np.array([0.25, 0.15, 0.05]),            # действие, похожее на "сидит"
}

# Рисуем 3D-график эмбеддингов
fig = plt.figure(figsize=(12, 8))                       # создаём фигуру
ax = fig.add_subplot(111, projection='3d')              # 3D-проекция

# Цвета по категориям
category_colors = {
    "кот": "red", "кошка": "red",                       # животные-коты - красные
    "собака": "orange", "щенок": "orange",              # животные-собаки - оранжевые
    "коврик": "blue", "диван": "blue",                  # предметы - синие
    "сидит": "green", "лежит": "green",                 # действия - зелёные
}

# Рисуем точки и подписи
for word, emb in toy_embeddings.items():                # для каждого слова
    color = category_colors[word]                       # цвет по категории
    ax.scatter(emb[0], emb[1], emb[2], c=color, s=200,  # рисуем точку
               edgecolor='black', linewidth=1, zorder=5)
    ax.text(emb[0]+0.03, emb[1]+0.03, emb[2]+0.03, word,  # подпись
            fontsize=12, fontweight='bold', color=color)

# Рисуем связи между похожими словами
pairs = [("кот", "кошка"), ("кот", "собака"), ("коврик", "диван"),  # пары
         ("сидит", "лежит"), ("кот", "сидит")]
for w1, w2 in pairs:                                   # для каждой пары
    e1, e2 = toy_embeddings[w1], toy_embeddings[w2]    # их эмбеддинги
    ax.plot([e1[0], e2[0]], [e1[1], e2[1]], [e1[2], e2[2]],  # линия
            'k--', alpha=0.3, linewidth=1)

# Вычисляем косинусную близость для примера
def cosine_similarity(v1, v2):
    # Косинусная близость: 1 = одинаковые, 0 = перпендикулярные, -1 = противоположные
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))  # формула

sim_cat_cat = cosine_similarity(toy_embeddings["кот"], toy_embeddings["кошка"])
sim_cat_dog = cosine_similarity(toy_embeddings["кот"], toy_embeddings["собака"])
sim_cat_carpet = cosine_similarity(toy_embeddings["кот"], toy_embeddings["коврик"])

ax.set_xlabel("Размер 1 (животность)", fontsize=11)    # подпись оси
ax.set_ylabel("Размер 2 (размер)", fontsize=11)         # подпись оси
ax.set_zlabel("Размер 3 (предметность)", fontsize=11)   # подпись оси
ax.set_title("Эмбеддинги: похожие слова - рядом в пространстве",  # заголовок
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Косинусная близость (1 = одинаковые, 0 = разные):")
print(f"  кот <-> кошка:  {sim_cat_cat:.3f}  <- очень похожи!")
print(f"  кот <-> собака: {sim_cat_dog:.3f}  <- похоже (оба животные)")
print(f"  кот <-> коврик: {sim_cat_carpet:.3f}  <- совсем разные!")
print()
print("В эмбеддингах расстояние между векторами отражает СМЫСЛОВУЮ близость")

---
## Шаг 8. Word2Vec: идея

**Word2Vec** (Mikolov et al., 2013) - алгоритм, который учит эмбеддинги из контекста.

**Ключевая идея**: слова, встречающиеся в похожем контексте, должны иметь похожие векторы.

> "Вы узнаёте компанию по тому, с кем он общается" - принцип дистрибутивной семантики

Две архитектуры:

| Архитектура | Что предсказываем | Идея |
|-------------|-------------------|------|
| **CBOW** (Continuous Bag of Words) | Центральное слово по контексту | По соседям угадываем слово |
| **Skip-gram** | Контекст по центральному слову | По слову угадываем соседей |

Пример CBOW:
```
Контекст: ["кот", "на"] -> Предсказываем: "сидит"
Контекст: ["сидит", "коврике"] -> Предсказываем: "на"
```

Мы реализуем **упрощённый Skip-gram** - чтобы понять механику!

In [ ]:
# ============================================================
# ШАГ 8a: Подготовка данных для Skip-gram
# ============================================================

def generate_skipgram_pairs(tokenized_docs, window_size=2):
    # Генерируем обучающие пары (целевое слово, контекстное слово).
    #
    # Skip-gram: по целевому слову предсказываем контекст.
    # Для каждого слова берём window_size слов слева и справа.
    #
    # Пример: "кот сидит на коврике", window=2
    # Целевое "сидит" -> Контекст: ["кот", "на"]
    # Целевое "на"    -> Контекст: ["сидит", "коврике"]
    #
    # Args:
    #   tokenized_docs: список списков токенов
    #   window_size: размер окна контекста
    # Returns:
    #   list: пары (целевое_слово, контекстное_слово)

    pairs = []                                         # пустой список пар
    for doc_tokens in tokenized_docs:                  # для каждого документа
        for i, target in enumerate(doc_tokens):        # для каждого слова (целевого)
            # Берём контекст слева
            for j in range(max(0, i - window_size), i):  # от i-window до i
                pairs.append((target, doc_tokens[j]))  # пара (целевое, контекстное)
            # Берём контекст справа
            for j in range(i + 1, min(len(doc_tokens), i + window_size + 1)):  # от i+1 до i+window
                pairs.append((target, doc_tokens[j]))  # пара (целевое, контекстное)
    return pairs                                       # возвращаем все пары


# Создаём расширенный корпус для обучения
w2v_corpus = [
    "кот сидит на коврике и мурлычет",
    "собака лежит на коврике и спит",
    "кот лежит на диване и мурлычет",
    "щенок играет на коврике и лает",
    "кошка сидит на диване и спит",
    "кот и собака играют вместе",
    "кот ест рыбу на кухне",
    "собака грызёт кость на коврике",
    "кошка спит на диване и мурлычет",
    "щенок бегает по коврику и лает"
]

# Токенизируем
tokenized_w2v = [tokenize_word(doc) for doc in w2v_corpus]  # токены

# Строим словарь
w2v_vocab, w2v_idx2token = build_vocabulary(tokenized_w2v)  # словари
vocab_size_w2v = len(w2v_vocab)                        # размер словаря

# Генерируем обучающие пары
skipgram_pairs = generate_skipgram_pairs(tokenized_w2v, window_size=2)  # пары

print(f"Размер словаря: {vocab_size_w2v} токенов")
print(f"Количество обучающих пар: {len(skipgram_pairs)}")
print(f"\nПервые 10 пар (целевое -> контекстное):")
for target, context in skipgram_pairs[:10]:            # показываем первые 10
    print(f"  {target:12s} -> {context}")

In [ ]:
# ============================================================
# ШАГ 8b: Обучаем простейший Skip-gram на numpy
# ============================================================

def train_skipgram(token2idx, pairs, embedding_dim=10, epochs=100, lr=0.05):
    # Обучаем Skip-gram модель с помощью SGD.
    #
    # Модель:
    # - W1: матрица (vocab_size x embedding_dim) - эмбеддинги целевых слов
    # - W2: матрица (embedding_dim x vocab_size) - веса контекстных слов
    # - Для пары (target, context): предсказываем P(context | target)
    #
    # Используем sigmoid + negative sampling (упрощённо).
    #
    # Args:
    #   token2idx: словарь токен -> индекс
    #   pairs: обучающие пары (target, context)
    #   embedding_dim: размерность эмбеддинга
    #   epochs: количество эпох
    #   lr: learning rate
    # Returns:
    #   np.array: матрица эмбеддингов (vocab_size x embedding_dim)

    V = len(token2idx)                                 # размер словаря
    D = embedding_dim                                  # размерность

    # Инициализируем матрицы маленькими случайными числами
    np.random.seed(42)                                 # фиксируем для воспроизводимости
    W1 = np.random.randn(V, D) * 0.1                   # эмбеддинги целевых слов
    W2 = np.random.randn(D, V) * 0.1                   # веса контекстных слов

    losses = []                                        # для отслеживания loss
    pairs_copy = list(pairs)                           # копия пар для перемешивания

    for epoch in range(epochs):                         # по эпохам
        total_loss = 0                                 # суммарный loss за эпоху

        # Перемешиваем пары
        np.random.shuffle(pairs_copy)                  # случайный порядок

        for target, context in pairs_copy:             # для каждой обучающей пары
            t_idx = token2idx[target]                  # индекс целевого слова
            c_idx = token2idx[context]                 # индекс контекстного слова

            # Forward pass
            h = W1[t_idx]                              # эмбеддинг целевого слова (1xD)
            score = np.dot(h, W2[:, c_idx])            # скалярное произведение с контекстом
            pred = 1 / (1 + np.exp(-score))            # sigmoid -> вероятность

            # Loss: бинарная кросс-энтропия
            loss = -np.log(pred + 1e-10)               # loss для положительного примера
            total_loss += loss                          # добавляем к сумме

            # Backward pass (градиенты)
            grad = pred - 1                             # градиент sigmoid x BCE

            # Обновляем W2 (контекстный вектор)
            W2[:, c_idx] -= lr * grad * h              # градиентный шаг для W2

            # Обновляем W1 (эмбеддинг целевого слова)
            W1[t_idx] -= lr * grad * W2[:, c_idx]      # градиентный шаг для W1

        # Сохраняем средний loss
        avg_loss = total_loss / len(pairs_copy)        # средний loss
        losses.append(avg_loss)                        # добавляем в список

        if (epoch + 1) % 20 == 0:                     # каждые 20 эпох
            print(f"  Эпоха {epoch+1:3d}: loss = {avg_loss:.4f}")  # логируем

    return W1, losses                                   # возвращаем эмбеддинги и loss


# Обучаем!
print("Обучаем Skip-gram Word2Vec...")
print(f"   Словарь: {vocab_size_w2v} слов, эмбеддинг: 10D, эпохи: 100")
print()

W1_embeddings, train_losses = train_skipgram(          # обучаем модель
    w2v_vocab, list(skipgram_pairs),                   # словарь и пары
    embedding_dim=10,                                   # размерность эмбеддинга
    epochs=100,                                         # количество эпох
    lr=0.05                                             # learning rate
)

print(f"\nОбучение завершено! Форма матрицы эмбеддингов: {W1_embeddings.shape}")

In [ ]:
# ============================================================
# ШАГ 8c: Визуализация процесса обучения
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))                 # создаём фигуру
ax.plot(range(1, len(train_losses)+1), train_losses,   # рисуем loss
        color='#FF6B6B', linewidth=2, label='Loss')
ax.set_xlabel("Эпоха", fontsize=13)                    # подпись оси X
ax.set_ylabel("Средний Loss", fontsize=13)              # подпись оси Y
ax.set_title("Обучение Skip-gram: loss уменьшается", fontsize=14, fontweight='bold')  # заголовок
ax.legend(fontsize=12)                                 # легенда
plt.tight_layout()
plt.show()

print("Loss уменьшается - модель учится предсказывать контекст по слову")
print("Эмбеддинги слов, которые встречаются в похожем контексте, становятся ближе")

---
## Шаг 9. Word2Vec: визуализация эмбеддингов

Наши эмбеддинги 10-мерные - мы не можем их увидеть напрямую.
Но мы можем **снизить размерность** до 2D с помощью **t-SNE** и увидеть кластеры похожих слов!

In [ ]:
# ============================================================
# ШАГ 9a: Снижаем размерность с t-SNE и визуализируем
# ============================================================

from sklearn.manifold import TSNE                      # алгоритм снижения размерности

# Снижаем 10D -> 2D для визуализации
tsne = TSNE(n_components=2, random_state=42,           # t-SNE с 2 компонентами
            perplexity=min(5, len(w2v_vocab)-1))       # perplexity (маленький словарь)
embeddings_2d = tsne.fit_transform(W1_embeddings)      # (V, 10) -> (V, 2)

# Создаём категории слов для раскраски
animal_words = {"кот", "кошка", "собака", "щенок"}    # животные
action_words = {"сидит", "лежит", "спит", "играет", "ест", "мурлычет", "лает", "грызёт", "бегает"}  # действия
place_words = {"коврике", "диване", "кухне"}           # места
other_words = {"на", "и", "по", "вместе", "рыбу", "кость"}  # прочие

def get_category(word):
    # Определяем категорию слова для раскраски
    if word in animal_words:
        return "Животные", "#FF6B6B"                  # красный
    elif word in action_words:
        return "Действия", "#4ECDC4"                  # бирюзовый
    elif word in place_words:
        return "Места", "#45B7D1"                     # синий
    else:
        return "Прочие", "#95A5A6"                    # серый

# Рисуем 2D-визуализацию
fig, ax = plt.subplots(figsize=(14, 10))               # создаём фигуру

# Рисуем точки по категориям
for i, (token, idx) in enumerate(w2v_vocab.items()):  # для каждого слова
    category, color = get_category(token)              # категория и цвет
    x, y = embeddings_2d[idx]                          # 2D-координаты
    ax.scatter(x, y, c=color, s=150, edgecolor='black',  # рисуем точку
               linewidth=1, zorder=5)
    ax.annotate(token, (x, y), fontsize=11, fontweight='bold',  # подпись
                xytext=(5, 5), textcoords='offset points', color=color)

# Добавляем легенду вручную
from matplotlib.patches import Patch                   # для легенды
legend_elements = [
    Patch(facecolor='#FF6B6B', edgecolor='black', label='Животные'),
    Patch(facecolor='#4ECDC4', edgecolor='black', label='Действия'),
    Patch(facecolor='#45B7D1', edgecolor='black', label='Места'),
    Patch(facecolor='#95A5A6', edgecolor='black', label='Прочие'),
]
ax.legend(handles=legend_elements, fontsize=12, loc='best')  # легенда

ax.set_title("Word2Vec эмбеддинги: слова с похожим контекстом - рядом",  # заголовок
             fontsize=14, fontweight='bold')
ax.set_xlabel("t-SNE размер 1", fontsize=12)           # подпись оси
ax.set_ylabel("t-SNE размер 2", fontsize=12)           # подпись оси
plt.tight_layout()
plt.show()

print("Животные (кот, кошка, собака) кластеризуются вместе")
print("Действия (сидит, лежит, спит) - рядом друг с другом")
print("Это доказывает: модель научилась различать семантику!")

In [ ]:
# ============================================================
# ШАГ 9b: Интерактивный поиск похожих слов
# ============================================================

def find_most_similar(word, embeddings, token2idx, idx2token, top_k=5):
    # Находим top_k самых похожих слов по косинусной близости.
    # Косинусная близость = cos(angle) = (A.B) / (|A|*|B|)
    #
    # Args:
    #   word: слово для поиска
    #   embeddings: матрица эмбеддингов
    #   token2idx: словарь токен -> индекс
    #   idx2token: словарь индекс -> токен
    #   top_k: сколько похожих слов вернуть
    # Returns:
    #   list: пары (слово, косинусная близость)

    if word not in token2idx:                           # если слова нет в словаре
        return []                                      # пустой список

    word_idx = token2idx[word]                          # индекс слова
    word_vec = embeddings[word_idx]                     # его эмбеддинг

    # Вычисляем косинусную близость со всеми словами
    similarities = []                                   # список (слово, близость)
    for idx in range(len(embeddings)):                 # для каждого слова
        if idx == word_idx:                             # пропускаем само слово
            continue
        vec = embeddings[idx]                           # эмбеддинг другого слова
        sim = cosine_similarity(word_vec, vec)          # косинусная близость
        similarities.append((idx2token[idx], sim))      # добавляем в список

    # Сортируем по убыванию близости
    similarities.sort(key=lambda x: -x[1])              # по убыванию

    return similarities[:top_k]                         # возвращаем top_k


# Интерактивный виджет для поиска похожих слов
word_dropdown = widgets.Dropdown(                       # выпадающий список
    options=list(w2v_vocab.keys()),                     # все слова словаря
    value='кот',                                        # выбранное по умолчанию
    description='Слово:',
    layout=widgets.Layout(width='250px')
)

def show_similar(word):
    # Показываем самые похожие слова с визуализацией
    similar = find_most_similar(word, W1_embeddings, w2v_vocab, w2v_idx2token, top_k=5)  # ищем

    if not similar:                                     # если ничего не нашли
        print(f"Слово '{word}' не найдено в словаре")
        return

    # Рисуем столбчатую диаграмму
    fig, ax = plt.subplots(figsize=(10, 5))             # фигура
    words = [s[0] for s in similar]                    # слова
    sims = [s[1] for s in similar]                     # близости
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(words)))  # цвета

    bars = ax.barh(words[::-1], sims[::-1], color=colors[::-1],  # горизонтальные столбцы
                   edgecolor='black', linewidth=1)
    ax.set_xlabel("Косинусная близость", fontsize=13)   # подпись
    ax.set_title(f"Слова, похожие на '{word}'", fontsize=14, fontweight='bold')  # заголовок
    ax.set_xlim(0, 1)                                  # пределы

    # Значения на столбцах
    for bar, sim in zip(bars, sims[::-1]):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{sim:.3f}', va='center', fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f"\nСлово: '{word}'")
    for i, (sim_word, sim_val) in enumerate(similar):  # выводим список
        print(f"  {i+1}. {sim_word:12s} -> {sim_val:.3f}")

# Связываем виджет
interactive_similar = interactive(show_similar, word=word_dropdown)  # интерактив
display(interactive_similar)                            # показываем

print("Выберите слово из списка - увидите самые похожие слова по эмбеддингам!")

---
## Шаг 10. BPE: Byte-Pair Encoding

**BPE** - алгоритм субворд-токенизации, который используют GPT, BERT и большинство современных LLM.

Идея:
1. Начинаем с **посимвольного** разбиения
2. Находим **самую частую пару** символов
3. **Объединяем** её в новый токен
4. Повторяем, пока не достигнем нужного размера словаря

Пример:
```
Шаг 0: ["к", "о", "т", " ", "с", "и", "д", "и", "т"]
Шаг 1: ["и", "т"] часто вместе -> объединяем в "ит"
Шаг 2: ["к", "о"] часто вместе -> объединяем в "ко"
...
Итог: ["ко", "т", "ит"] - более компактно!
```

Почему BPE круто:
- Баланс между word-level и char-level
- Может обработать **любое** слово (разбивает неизвестное на подслова)
- Частые слова = целые токены, редкие = комбинации подслов

In [ ]:
# ============================================================
# ШАГ 10a: BPE - алгоритм обучения с нуля
# ============================================================

class BPETokenizer:
    # Byte-Pair Encoding токенизатор - реализация с нуля.
    #
    # Алгоритм обучения:
    # 1. Начинаем с посимвольного разбиения всех слов
    # 2. На каждом шаге находим самую частую пару токенов
    # 3. Объединяем эту пару в новый токен
    # 4. Добавляем в словарь объединений
    # 5. Повторяем num_merges раз
    #
    # Токенизация нового слова:
    # 1. Разбиваем на символы
    # 2. Последовательно применяем объединения в порядке приоритета

    def __init__(self):
        # Инициализируем пустой токенизатор
        self.merges = []                               # список объединений (пара -> новый токен)
        self.vocab = set()                              # словарь токенов

    def train(self, texts, num_merges=50, verbose=True):
        # Обучаем BPE на корпусе текстов.
        #
        # Args:
        #   texts (list): список строк (корпус)
        #   num_merges (int): количество шагов объединения
        #   verbose (bool): печатать ли прогресс

        # Шаг 1: разбиваем все слова на символы + спецсимвол конца слова
        word_splits = {}                                # словарь: слово -> разбиение на символы
        word_counts = Counter()                         # частоты слов

        for text in texts:                              # для каждого текста
            words = text.lower().split()                # разбиваем на слова
            for word in words:                          # для каждого слова
                word_clean = re.sub(r'[^\w]', '', word)  # убираем пунктуацию
                if word_clean:                          # если не пустое
                    word_counts[word_clean] += 1        # увеличиваем счётчик
                    # Символьное разбиение с маркером конца слова
                    word_splits[word_clean] = list(word_clean) + ['</w>']  # "кот" -> ["к","о","т","</w>"]

        # Инициализируем словарь всеми символами
        self.vocab = set()                              # пустое множество
        for splits in word_splits.values():             # для каждого разбиения
            self.vocab.update(splits)                   # добавляем все символы

        # Шаг 2: повторяем num_merges раз
        for merge_idx in range(num_merges):             # для каждого шага
            # Считаем частоты всех пар
            pair_counts = Counter()                     # счётчик пар
            for word, count in word_counts.items():     # для каждого слова
                splits = word_splits[word]              # текущее разбиение
                for i in range(len(splits) - 1):       # для каждой позиции
                    pair = (splits[i], splits[i+1])     # пара соседних токенов
                    pair_counts[pair] += count          # увеличиваем частоту

            if not pair_counts:                         # если пар не осталось
                break                                   # заканчиваем

            # Находим самую частую пару
            best_pair = pair_counts.most_common(1)[0][0]  # самая частая пара

            # Объединяем эту пару во всех словах
            new_token = best_pair[0] + best_pair[1]     # новый токен = конкатенация
            self.merges.append(best_pair)               # запоминаем объединение
            self.vocab.add(new_token)                   # добавляем в словарь

            # Применяем объединение ко всем словам
            for word in word_splits:                    # для каждого слова
                new_splits = []                         # новое разбиение
                i = 0                                   # индекс
                splits = word_splits[word]              # текущее разбиение
                while i < len(splits):                  # пока не конец
                    if (i < len(splits) - 1 and         # если есть следующий
                        splits[i] == best_pair[0] and   # и текущий = первый элемент пары
                        splits[i+1] == best_pair[1]):   # и следующий = второй элемент пары
                        new_splits.append(new_token)    # объединяем в новый токен
                        i += 2                          # пропускаем два символа
                    else:
                        new_splits.append(splits[i])    # оставляем как есть
                        i += 1                          # следующий символ
                word_splits[word] = new_splits          # обновляем разбиение

            if verbose and merge_idx < 10:              # логируем первые 10
                print(f"  Шаг {merge_idx+1:2d}: '{best_pair[0]}' + '{best_pair[1]}' -> '{new_token}' "
                      f"(частота: {pair_counts[best_pair]})")

        if verbose:
            print(f"\n  Обучено {len(self.merges)} объединений")
            print(f"  Размер словаря: {len(self.vocab)} токенов")

    def tokenize(self, text):
        # Токенизируем текст обученным BPE.
        #
        # Args:
        #   text (str): текст для токенизации
        # Returns:
        #   list: список BPE-токенов

        words = text.lower().split()                    # разбиваем на слова
        all_tokens = []                                 # все токены

        for word in words:                              # для каждого слова
            word_clean = re.sub(r'[^\w]', '', word)    # убираем пунктуацию
            if not word_clean:                          # если пустое
                continue                                # пропускаем

            # Начальное разбиение: по символам
            splits = list(word_clean) + ['</w>']        # посимвольно + маркер конца

            # Применяем объединения по порядку приоритета
            for pair in self.merges:                    # для каждого объединения
                new_splits = []                         # новое разбиение
                i = 0                                   # индекс
                while i < len(splits):                  # пока не конец
                    if (i < len(splits) - 1 and         # если есть следующий
                        splits[i] == pair[0] and        # совпадает первый
                        splits[i+1] == pair[1]):        # совпадает второй
                        new_splits.append(pair[0] + pair[1])  # объединяем
                        i += 2                          # пропускаем два
                    else:
                        new_splits.append(splits[i])    # оставляем
                        i += 1                          # следующий
                splits = new_splits                     # обновляем

            all_tokens.extend(splits)                   # добавляем токены слова

        return all_tokens                               # возвращаем все токены


# === Обучаем BPE на нашем корпусе ===
print("Обучаем BPE-токенизатор...")
print()

bpe = BPETokenizer()                                   # создаём токенизатор
bpe.train(w2v_corpus, num_merges=30, verbose=True)     # обучаем на корпусе

In [ ]:
# ============================================================
# ШАГ 10b: Тестируем BPE-токенизатор
# ============================================================

# Токенизируем несколько предложений
test_sentences = [
    "кот сидит на коврике",
    "кошка спит на диване",
    "собака грызёт кость",                               # слово "кость" могло не быть в обучении
    "программирование это интересно"                     # полностью новые слова
]

print("BPE-токенизация тестовых предложений:")
print("=" * 60)

for sent in test_sentences:                              # для каждого предложения
    tokens = bpe.tokenize(sent)                         # токенизируем
    print(f"\nТекст:    {sent}")
    print(f"   Токены:   {tokens}")
    print(f"   Кол-во:   {len(tokens)} токенов")

print()
print("Заметьте:")
print("   - Частые слова ('кот', 'сидит') - целые токены или почти")
print("   - Редкие слова разбиваются на подслова")
print("   - Неизвестные слова разбиваются на символы - НЕТ ошибки OOV!")
print("   - </w> отмечает конец слова")

In [ ]:
# ============================================================
# ШАГ 10c: Визуализация BPE-объединений
# ============================================================

# Показываем первые 15 объединений как дерево
fig, ax = plt.subplots(figsize=(14, 8))                 # создаём фигуру

# Берём первые 15 объединений
num_show = min(15, len(bpe.merges))                     # сколько показать

for i in range(num_show):                               # для каждого объединения
    pair = bpe.merges[i]                                # пара токенов
    new_token = pair[0] + pair[1]                       # результат объединения

    y_pos = num_show - 1 - i                            # позиция по Y (сверху вниз)

    # Левый токен
    ax.add_patch(plt.Rectangle((0, y_pos - 0.3), 3, 0.6,  # прямоугольник
                               facecolor='#FF6B6B', edgecolor='black', linewidth=1.5))
    ax.text(1.5, y_pos, pair[0], ha='center', va='center', fontsize=10, fontweight='bold')

    # Знак +
    ax.text(3.5, y_pos, '+', ha='center', va='center', fontsize=14, fontweight='bold')

    # Правый токен
    ax.add_patch(plt.Rectangle((4, y_pos - 0.3), 3, 0.6,  # прямоугольник
                               facecolor='#4ECDC4', edgecolor='black', linewidth=1.5))
    ax.text(5.5, y_pos, pair[1], ha='center', va='center', fontsize=10, fontweight='bold')

    # Стрелка ->
    ax.annotate('', xy=(8, y_pos), xytext=(7.5, y_pos),  # стрелка
                arrowprops=dict(arrowstyle='->', lw=2, color='#2C3E50'))

    # Результат
    ax.add_patch(plt.Rectangle((8, y_pos - 0.3), 4, 0.6,  # прямоугольник
                               facecolor='#F39C12', edgecolor='black', linewidth=1.5))
    ax.text(10, y_pos, new_token, ha='center', va='center', fontsize=10, fontweight='bold')

    # Номер шага
    ax.text(-0.5, y_pos, f'{i+1}', ha='center', va='center', fontsize=11,
            fontweight='bold', color='gray')

ax.set_xlim(-1, 13)                                    # пределы
ax.set_ylim(-1, num_show)                              # пределы
ax.set_title("BPE: шаги объединения токенов", fontsize=14, fontweight='bold')  # заголовок
ax.axis('off')                                          # убираем оси
plt.tight_layout()
plt.show()

print("На каждом шаге самая частая пара объединяется в новый токен")
print("Частые комбинации символов становятся подсловами")

---
## Шаг 11. BPE: интерактивный демонстратор

Попробуйте менять текст и количество объединений - увидите, как BPE адаптируется!

In [ ]:
# ============================================================
# ШАГ 11: Интерактивный BPE-демонстратор
# ============================================================

def interactive_bpe(num_merges, text):
    # Интерактивный BPE: меняем параметры и видим результат.
    #
    # Args:
    #   num_merges (int): количество объединений BPE
    #   text (str): текст для токенизации

    if not text.strip():                                # если пустой текст
        print("Введите текст!")
        return

    # Обучаем BPE с заданным числом объединений
    bpe_interactive = BPETokenizer()                    # создаём новый токенизатор
    bpe_interactive.train(w2v_corpus + [text],          # обучаем на корпусе + новый текст
                          num_merges=num_merges, verbose=False)  # без вывода

    # Токенизируем
    tokens = bpe_interactive.tokenize(text)             # токены

    # Также показываем word-level и char-level для сравнения
    word_tokens = tokenize_word(text)                   # word-level
    char_tokens = list(text.replace(' ', '_'))         # char-level (пробел = _)

    # Визуализация
    fig, axes = plt.subplots(3, 1, figsize=(16, 7))    # 3 строки

    # --- Char-level ---
    ax = axes[0]
    for i, ch in enumerate(char_tokens[:30]):           # первые 30 символов
        color = '#E8D5B7' if ch != '_' else '#BDC3C7'  # цвет
        ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, facecolor=color,  # прямоугольник
                                   edgecolor='black', linewidth=1))
        ax.text(i + 0.45, 0.5, ch, ha='center', va='center', fontsize=9)
    ax.set_xlim(-0.1, min(30, len(char_tokens)))        # пределы
    ax.set_ylim(-0.2, 1.3)
    ax.set_title(f"Char-level: {len(char_tokens)} токенов", fontsize=12, fontweight='bold')
    ax.axis('off')

    # --- BPE ---
    ax = axes[1]
    colors_bpe = plt.cm.Set2(np.linspace(0, 1, max(len(tokens), 1)))  # цвета
    for i, token in enumerate(tokens[:20]):              # первые 20 токенов
        ax.add_patch(plt.Rectangle((i * 1.5, 0), 1.4, 1, facecolor=colors_bpe[i % len(colors_bpe)],  # прямоугольник
                                   edgecolor='black', linewidth=1.5))
        ax.text(i * 1.5 + 0.7, 0.5, token, ha='center', va='center',  # текст
                fontsize=9, fontweight='bold')
    ax.set_xlim(-0.1, min(20, len(tokens)) * 1.5)      # пределы
    ax.set_ylim(-0.2, 1.3)
    ax.set_title(f"BPE ({num_merges} объединений): {len(tokens)} токенов",  # заголовок
                 fontsize=12, fontweight='bold', color='#E74C3C')
    ax.axis('off')

    # --- Word-level ---
    ax = axes[2]
    colors_word = plt.cm.Set3(np.linspace(0, 1, max(len(word_tokens), 1)))  # цвета
    for i, token in enumerate(word_tokens[:15]):         # первые 15 слов
        ax.add_patch(plt.Rectangle((i * 2, 0), 1.9, 1, facecolor=colors_word[i % len(colors_word)],  # прямоугольник
                                   edgecolor='black', linewidth=1.5))
        ax.text(i * 2 + 0.95, 0.5, token, ha='center', va='center',  # текст
                fontsize=10, fontweight='bold')
    ax.set_xlim(-0.1, min(15, len(word_tokens)) * 2)   # пределы
    ax.set_ylim(-0.2, 1.3)
    ax.set_title(f"Word-level: {len(word_tokens)} токенов", fontsize=12, fontweight='bold')
    ax.axis('off')

    plt.suptitle("Сравнение токенизации: Char vs BPE vs Word", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Текстовый вывод
    print(f"\nСравнение для текста: '{text}'")
    print(f"  Char-level: {len(char_tokens)} токенов")
    print(f"  BPE:        {len(tokens)} токенов  <- с {num_merges} объединениями")
    print(f"  Word-level: {len(word_tokens)} токенов")
    print(f"\n  BPE токены: {tokens}")


# Создаём виджеты
text_widget = widgets.Text(                              # текстовое поле
    value='кот сидит на диване',
    description='Текст:',
    layout=widgets.Layout(width='400px')
)

merges_slider = widgets.IntSlider(                       # слайдер объединений
    value=20,                                           # по умолчанию
    min=5,                                              # минимум
    max=50,                                             # максимум
    step=5,                                             # шаг
    description='Объед.:',
    continuous_update=False                              # обновлять только при отпускании
)

# Связываем виджеты
interactive_bpe_widget = interactive(interactive_bpe,    # интерактивная функция
                                     num_merges=merges_slider,  # слайдер
                                     text=text_widget)   # текстовое поле
display(interactive_bpe_widget)                         # показываем

print("Меняйте текст и количество объединений - увидите, как BPE адаптируется!")

---
## Шаг 12. Сравнение подходов

Теперь у нас есть четыре способа превратить текст в числа. Давайте сравним их!

In [ ]:
# ============================================================
# ШАГ 12a: Сводная таблица сравнения подходов
# ============================================================

# Создаём красивую таблицу сравнения
comparison_data = {
    "Метод": ["One-hot", "TF-IDF", "Word2Vec", "BPE"],
    "Тип вектора": ["Разреженный", "Разреженный", "Плотный", "Подслова"],
    "Размерность": [f"|V| = {len(w2v_vocab)}", f"|V| = {len(w2v_vocab)}", "10 (100-300 типично)", "Var"],
    "Сходство слов": ["Нет", "Частичное", "Да", "Частичное"],
    "OOV слова": ["Ошибка", "Ошибка", "Ошибка", "Разбивает"],
    "Контекст": ["Нет", "Документ", "Окно", "Нет"],
    "Применение": ["Базовый", "Поиск/классиф.", "NLP задачи", "LLM токенизация"],
}

# Выводим таблицу
html = "<table style='border-collapse:collapse; width:100%; font-size:14px'>"
html += "<tr style='background:#2C3E50; color:white'>"
for header in comparison_data.keys():                   # заголовки
    html += f"<th style='padding:10px; border:1px solid #ddd'>{header}</th>"
html += "</tr>"

for row_idx in range(len(comparison_data["Метод"])):   # строки
    bg = "#F8F9FA" if row_idx % 2 == 0 else "white"    # чередуем цвет
    html += f"<tr style='background:{bg}'>"
    for col_idx, key in enumerate(comparison_data.keys()):  # ячейки
        val = comparison_data[key][row_idx]             # значение
        if col_idx == 0:                                # первый столбец - жирный
            html += f"<td style='padding:8px; border:1px solid #ddd; font-weight:bold'>{val}</td>"
        else:
            html += f"<td style='padding:8px; border:1px solid #ddd; text-align:center'>{val}</td>"
    html += "</tr>"

html += "</table>"
display(HTML(html))                                     # показываем HTML-таблицу

print()
print("Ключевые выводы:")
print("  1. One-hot - самый простой, но бесполезный для NLP")
print("  2. TF-IDF - хорош для поиска и классификации документов")
print("  3. Word2Vec - первый метод, дающий плотные эмбеддинги с семантикой")
print("  4. BPE - стандарт для современных LLM (GPT, BERT)")

In [ ]:
# ============================================================
# ШАГ 12b: Визуальное сравнение: размерность vs качество
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))        # три графика

# --- График 1: Размерность вектора ---
ax = axes[0]
methods = ["One-hot", "TF-IDF", "Word2Vec", "BPE"]
dims = [len(w2v_vocab), len(w2v_vocab), 10, 8]        # размерности
colors = ['#E74C3C', '#F39C12', '#2ECC71', '#3498DB']  # цвета
bars = ax.bar(methods, dims, color=colors, edgecolor='black', linewidth=1.2)  # столбцы
ax.set_ylabel("Размерность", fontsize=12)
ax.set_title("Размерность вектора", fontsize=13, fontweight='bold')
for bar, dim in zip(bars, dims):                       # подписи
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(dim), ha='center', fontsize=11, fontweight='bold')

# --- График 2: Качество передачи смысла ---
ax = axes[1]
quality = [0, 3, 9, 5]                                 # условные баллы 0-10
bars = ax.bar(methods, quality, color=colors, edgecolor='black', linewidth=1.2)
ax.set_ylabel("Качество семантики (0-10)", fontsize=12)
ax.set_title("Передача смысла", fontsize=13, fontweight='bold')
ax.set_ylim(0, 10)
for bar, q in zip(bars, quality):                      # подписи
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(q), ha='center', fontsize=11, fontweight='bold')

# --- График 3: Обработка неизвестных слов ---
ax = axes[2]
oov_handling = [0, 0, 2, 9]                            # условные баллы
bars = ax.bar(methods, oov_handling, color=colors, edgecolor='black', linewidth=1.2)
ax.set_ylabel("OOV устойчивость (0-10)", fontsize=12)
ax.set_title("Обработка неизвестных слов", fontsize=13, fontweight='bold')
ax.set_ylim(0, 10)
for bar, oov in zip(bars, oov_handling):               # подписи
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(oov), ha='center', fontsize=11, fontweight='bold')

plt.suptitle("Сравнение подходов к представлению текста", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Нет одного 'лучшего' метода - каждый подходит для своих задач")
print("Современный подход: BPE для токенизации + Transformer для эмбеддингов = GPT/BERT")

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

> Помните: вы - автор. Меняйте код, ломайте, чините. Это ваш ноутбук!

In [ ]:
# ============================================================
# ЗАДАНИЕ 1: Улучшите TF-IDF - добавьте стоп-слова
# ============================================================

# Стоп-слова - это частые слова, не несущие смысла (предлоги, союзы, местоимения)
# Добавьте фильтрацию стоп-слов в функцию compute_tf()

STOP_WORDS = {"на", "и", "в", "с", "по", "за", "из", "от", "это", "то",  # русский стоп-лист
              "а", "но", "не", "как", "так", "что", "кто", "где", "когда"}

def compute_tf_no_stopwords(tokens, stop_words=None):
    # TF без стоп-слов.
    #
    # TODO: Допишите функцию!
    # Подсказка: отфильтруйте токены из stop_words перед подсчётом
    #
    # Args:
    #   tokens: список токенов
    #   stop_words: множество стоп-слов
    # Returns:
    #   dict: токен -> TF (без стоп-слов)

    # Ваш код здесь:
    # 1. Отфильтруйте стоп-слова из tokens
    # 2. Посчитайте TF как в compute_tf()

    pass  # удалите эту строку и напишите свой код


# Проверьте: в результатах не должно быть "на", "и" и т.д.
# Раскомментируйте код ниже после написания функции:
# tf_clean = compute_tf_no_stopwords(tokenized_corpus[0], STOP_WORDS)
# print("TF без стоп-слов:", tf_clean)

print("ЗАДАНИЕ: допишите функцию compute_tf_no_stopwords() выше")
print("   Подсказка: используйте list comprehension или filter()")

In [ ]:
# ============================================================
# ЗАДАНИЕ 2: Исследуйте Word2Vec - аналогии слов
# ============================================================

# Знаменитый пример: king - man + woman ~ queen
# Попробуйте найти аналогии в наших эмбеддингах!

def word_analogy(word_a, word_b, word_c, embeddings, token2idx, idx2token, top_k=5):
    # Поиск аналогий: A относится к B как C относится к ?
    #
    # Векторная арифметика: result = B - A + C
    # Ищем ближайшее слово к result.
    #
    # Args:
    #   word_a, word_b, word_c: слова для аналогии
    #   embeddings: матрица эмбеддингов
    #   token2idx, idx2token: словари
    #   top_k: сколько результатов вернуть

    # Проверяем, что все слова есть в словаре
    for w in [word_a, word_b, word_c]:
        if w not in token2idx:
            print(f"Слово '{w}' не в словаре!")
            return

    # Векторная аналогия: B - A + C
    vec_a = embeddings[token2idx[word_a]]               # вектор A
    vec_b = embeddings[token2idx[word_b]]               # вектор B
    vec_c = embeddings[token2idx[word_c]]               # вектор C
    result_vec = vec_b - vec_a + vec_c                  # результат аналогии

    # Ищем ближайшие слова
    similarities = []
    for idx in range(len(embeddings)):
        if idx in [token2idx[word_a], token2idx[word_b], token2idx[word_c]]:
            continue                                   # пропускаем сами слова
        sim = cosine_similarity(result_vec, embeddings[idx])
        similarities.append((idx2token[idx], sim))

    similarities.sort(key=lambda x: -x[1])

    print(f"Аналогия: {word_a} -> {word_b} :: {word_c} -> ?")
    for i, (word, sim) in enumerate(similarities[:top_k]):
        print(f"  {i+1}. {word:12s} (similarity: {sim:.3f})")


# Попробуйте! (результаты могут быть неидеальными - маленький корпус)
print("ЗАДАНИЕ: попробуйте разные аналогии!")
print("   Пример: word_analogy('кот', 'коврике', 'собака', W1_embeddings, w2v_vocab, w2v_idx2token)")
print()

# Раскомментируйте и экспериментируйте:
# word_analogy('кот', 'коврике', 'собака', W1_embeddings, w2v_vocab, w2v_idx2token)
# word_analogy('сидит', 'кот', 'лежит', W1_embeddings, w2v_vocab, w2v_idx2token)

In [ ]:
# ============================================================
# ЗАДАНИЕ 3: Обучите BPE на большем корпусе
# ============================================================

# Попробуйте обучить BPE на более разнообразном тексте
# и посмотреть, как меняется токенизация

bigger_corpus = [
    "машинное обучение это раздел искусственного интеллекта",
    "глубокое обучение использует многослойные нейронные сети",
    "обработка естественного языка это применение машинного обучения",
    "компьютерное зрение распознаёт объекты на изображениях",
    "рекуррентные нейронные сети работают с последовательностями",
    "трансформеры заменили рекуррентные сети в задачах языка",
    "механизм внимания позволяет модели фокусироваться на важных частях",
    "эмбеддинги представляют слова как векторы в пространстве",
    "токенизация разбивает текст на минимальные значимые единицы",
    "байт пар энкодинг популярный алгоритм субворд токенизации"
]

# Ваш код:
# 1. Создайте новый BPE-токенизатор
# 2. Обучите на bigger_corpus с разным числом объединений (10, 30, 50)
# 3. Сравните результаты токенизации

# Создаём и обучаем
bpe_big = BPETokenizer()                                # новый токенизатор
bpe_big.train(bigger_corpus, num_merges=30, verbose=True)  # обучаем

# Тестируем
test = "нейронные сети и машинное обучение"
tokens_big = bpe_big.tokenize(test)                     # токенизируем
print(f"\nТекст:  {test}")
print(f"   Токены: {tokens_big}")

print("\nЗАДАНИЕ: обучите с num_merges=10 и num_merges=50 и сравните!")
print("   Как больше объединений влияет на токенизацию?")

In [ ]:
# ============================================================
# ЗАДАНИЕ 4: Реализуйте CBOW (Continuous Bag of Words)
# ============================================================

# CBOW - вторая архитектура Word2Vec
# В отличие от Skip-gram (по слову -> предсказываем контекст),
# CBOW по контексту -> предсказываем центральное слово

def generate_cbow_pairs(tokenized_docs, window_size=2):
    # Генерируем пары (контекст, целевое слово) для CBOW.
    #
    # TODO: Напишите эту функцию!
    # Подсказка: для каждого слова соберите его соседей (контекст)
    # и создайте пару (список_соседей, целевое_слово)
    #
    # Args:
    #   tokenized_docs: список списков токенов
    #   window_size: размер окна
    # Returns:
    #   list: пары (контекст, целевое_слово)

    # Ваш код здесь:
    pass  # удалите и напишите свой код


# Проверьте:
# cbow_data = generate_cbow_pairs(tokenized_w2v, window_size=2)
# print(f"CBOW пар: {len(cbow_data)}")
# for ctx, target in cbow_data[:5]:
#     print(f"  Контекст: {ctx} -> Целевое: {target}")

print("ЗАДАНИЕ: реализуйте generate_cbow_pairs()")
print("   Подсказка: это 'обратный' Skip-gram - собираем соседей, а не генерируем пары")

In [ ]:
# ============================================================
# ЗАДАНИЕ 5: Визуализируйте эмбеддинги для нового корпуса
# ============================================================

# Создайте свой корпус (минимум 15 предложений на любую тему!)
# Обучите Skip-gram Word2Vec и визуализируйте с t-SNE

# Ваш код здесь:
my_corpus = [
    # Напишите свои предложения (минимум 15!)
    # Чем разнообразнее - тем интереснее будут кластеры
]

# Шаги:
# 1. Токенизируйте my_corpus
# 2. Постройте словарь
# 3. Сгенерируйте Skip-gram пары
# 4. Обучите модель
# 5. Снизьте размерность t-SNE
# 6. Нарисуйте график

print("ЗАДАНИЕ: создайте корпус, обучите Word2Vec и визуализируйте!")
print("   Темы для корпуса: спорт, музыка, наука, игры - что угодно!")
print()
print("=" * 60)
print("Поздравляем! Вы прошли блокнот 'Обработка текста'")
print("=" * 60)
print()
print("Что вы освоили:")
print("  + Токенизация: word-level, char-level, subword")
print("  + One-hot кодирование и его проблемы")
print("  + TF-IDF: от формулы до реализации")
print("  + Word2Vec Skip-gram: от идеи до обученных эмбеддингов")
print("  + BPE: алгоритм, который используют GPT и BERT")
print("  + Сравнение подходов к представлению текста")
print()
print("Следующий блокнот: 06_rnn_lstm.ipynb -> Рекуррентные сети")